## Extracción de Caracteristicas para los sistemas trifasicos


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, lfilter, filtfilt
from scipy.fft import rfft, rfftfreq
from scipy.stats import skew, kurtosis, entropy
import pywt
from typing import List, Tuple, Dict, Any
import os
import glob
import nolds
from joblib import Parallel, delayed
import algoritmos_deteccion as ad

## Segmentación para sistemas trifasicos

In [ ]:
def _setup_academic_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['mathtext.fontset'] = 'stix'
    plt.rcParams['font.size'] = 14
    plt.rcParams['axes.labelsize'] = 16
    plt.rcParams['axes.titlesize'] = 18
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    plt.rcParams['legend.fontsize'] = 12
    plt.rcParams['lines.linewidth'] = 1.5
    

def visualizar_segmentacion_completa(va, vb, vc, fs, signal_id):
    """
    Genera un gráfico de 3 paneles para visualizar el proceso de segmentación.
    """
    _setup_academic_style()
    
    t = np.arange(len(va)) / fs

    # --- Generar máscaras para visualización (usando Fase A como referencia) ---
    mascara_rms = ad.generar_mascara_rms(va, fs)
    mascara_swt = ad.generar_mascara_swt(va)
    mascara_hilbert = ad.generar_mascara_hilbert(va)

    # --- Encontrar el segmento final trifásico ---
    t_inicio_idx, t_fin_idx = ad.encontrar_intervalo_global(va, vb, vc, fs)

    # --- Crear figura de 3 paneles ---
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    fig.suptitle(f'Proceso de Segmentación para: {signal_id}', fontsize=20)

    # Panel 1: Señal Trifásica Original
    axes[0].plot(t, va, label='Fase A')
    axes[0].plot(t, vb, label='Fase B')
    axes[0].plot(t, vc, label='Fase C')
    axes[0].set_ylabel('Amplitud (p.u.)')
    axes[0].set_title('Señal Original')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, linestyle=':')

    # Panel 2: Máscaras de Detección (sobre Fase A)
    axes[1].plot(t, va, color='gray', alpha=0.5, label='Fase A (ref.)')
    axes[1].fill_between(t, -1.5, 1.5, where=mascara_rms, alpha=0.4, label='Máscara RMS', step='post')
    axes[1].fill_between(t, -1.5, 1.5, where=mascara_hilbert, alpha=0.4, label='Máscara Hilbert', step='post')
    axes[1].fill_between(t, -1.5, 1.5, where=mascara_swt, alpha=0.4, label='Máscara SWT', step='post')
    axes[1].set_ylabel('Detección')
    axes[1].set_title('Máscaras Individuales (sobre Fase A)')
    axes[1].set_ylim(-1.5, 1.5)
    axes[1].legend(loc='upper right')
    axes[1].grid(True, linestyle=':')

    # Panel 3: Resultado de la Segmentación
    axes[2].plot(t, va, color='gray', alpha=0.3)
    axes[2].plot(t, vb, color='gray', alpha=0.3)
    axes[2].plot(t, vc, color='gray', alpha=0.3)
    if t_inicio_idx is not None and t_fin_idx is not None:
        axes[2].axvspan(t[t_inicio_idx], t[t_fin_idx], color='red', alpha=0.2, label='Segmento Global')
        t_segmento = t[t_inicio_idx:t_fin_idx]
        axes[2].plot(t_segmento, va[t_inicio_idx:t_fin_idx], color='blue', label='Fase A Seg.')
        axes[2].plot(t_segmento, vb[t_inicio_idx:t_fin_idx], color='orange', label='Fase B Seg.')
        axes[2].plot(t_segmento, vc[t_inicio_idx:t_fin_idx], color='green', label='Fase C Seg.')
    axes[2].set_xlabel('Tiempo (s)')
    axes[2].set_ylabel('Amplitud (p.u.)')
    axes[2].set_title('Resultado de Segmentación Trifásica')
    axes[2].legend(loc='upper right')
    axes[2].grid(True, linestyle=':')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# ==============================================================================
file_path = r"C:\Users\Juan_Saa\Documents\Juan_Saa\Proyecto_Grado\Power-Quality\Generacion_Señales\Trifasico\Dataset_Trifasico_Completo_2.parquet"
signal_id_prueba = "Sag_A_clean_0010"
fs = 7200 # frecuencia de muestreo

print(f"Buscando y cargando eficientemente la señal: {signal_id_prueba}...")

# 2. Carga SOLO la señal necesaria usando filtros (evita error de memoria)
try:
    filters = [('signal_id', '=', signal_id_prueba)]
    una_señal_df = pd.read_parquet(file_path, filters=filters)
except Exception as e:
    print(f"No se pudo cargar la señal. Error: {e}")
    # Salir si el DataFrame no se pudo crear
    exit()

# 3. El resto de tu código funciona igual
va_original = una_señal_df['Phase_A'].values
vb_original = una_señal_df['Phase_B'].values
vc_original = una_señal_df['Phase_C'].values

print(f"Procesando señal: {signal_id_prueba}")
print(f"Longitud original de la señal: {len(va_original)} muestras")

# Llama a la nueva función orquestadora
# (Asegúrate de que 'ad' esté importado si tus funciones están en otro archivo)
va_seg, vb_seg, vc_seg = ad.segmentar_evento_trifasico(va_original, vb_original, vc_original, fs)

print(f"Longitud de la señal segmentada: {len(va_seg)} muestras")

visualizar_segmentacion_completa(va_original, vb_original, vc_original, fs, signal_id_prueba)

## Caracteristicas

In [ ]:
# ==============================================================================
# 1. Dominio del Tiempo
# ==============================================================================
def caracteristicas_tiempo(segmento: np.ndarray, rms_nominal: float) -> dict:
    """Extrae las características seleccionadas del dominio del tiempo."""
    
    rms_segmento_val = np.sqrt(np.mean(segmento**2))
    profundidad_relativa_val = rms_segmento_val / rms_nominal if rms_nominal > 1e-6 else 0.0
    pico_abs = np.max(np.abs(segmento))
    factor_cresta_val = pico_abs / rms_segmento_val if rms_segmento_val > 1e-6 else 0.0
    curtosis_val = kurtosis(segmento)
    
    return {
        'rms_segmento': rms_segmento_val,
        'profundidad_relativa': profundidad_relativa_val,
        'factor_cresta': factor_cresta_val,
        'curtosis': curtosis_val,
    }

# ==============================================================================
# 2. Dominio de la Frecuencia 
# ==============================================================================
def caracteristicas_frecuencia(segmento: np.ndarray) -> dict:
    """Extrae únicamente la energía espectral (FFT)."""
    n = len(segmento)
    if n < 2: return {'fft_energia': 0}

    fft_vals = np.fft.fft(segmento)[:n // 2]
    psd = np.abs(fft_vals)**2 / n
    energia_espectral = np.sum(psd)
    
    return {
        'fft_energia': energia_espectral,
    }

# ==============================================================================
# 3. Dominio Wavelet y Envolvente 
# ==============================================================================
def caracteristicas_wavelet(segmento: np.ndarray, wavelet: str = 'db4', max_level: int = 4) -> dict:
    """Extrae energía, entropía y desviación estándar de los coeficientes wavelet."""
    coeffs = pywt.wavedec(segmento, wavelet, level=max_level)
    caracteristicas = {}
    
    for i in range(1, len(coeffs)):
        nivel_coeffs = coeffs[-i]
        nivel = i
        
        caracteristicas[f'wavelet_energia_cD{nivel}'] = np.sum(nivel_coeffs**2)
        
        energia = caracteristicas[f'wavelet_energia_cD{nivel}']
        p = (nivel_coeffs**2) / energia if energia > 1e-6 else 0
        caracteristicas[f'wavelet_entropia_cD{nivel}'] = -np.sum(p * np.log2(p + 1e-12))
        
        caracteristicas[f'wavelet_std_cD{nivel}'] = np.std(nivel_coeffs)
        
    return caracteristicas

def caracteristicas_envolvente(senal: np.ndarray) -> dict:
    """Calcula la media y desviación estándar de la envolvente de Hilbert."""
    envolvente = np.abs(hilbert(senal))
    return {
        'env_media': np.mean(envolvente),
        'env_std': np.std(envolvente)
    }

# ==============================================================================
# 4. FUNCIÓN MAESTRA 
# ==============================================================================
def extraer_caracteristicas(segmento_evento: np.ndarray, fs: int, rms_nominal_val: float) -> dict:

    caracteristicas = {}
    
    # Llama a las versiones ajustadas y a las que no necesitaron cambios
    caracteristicas.update(caracteristicas_tiempo(segmento_evento, rms_nominal_val))
    caracteristicas.update(caracteristicas_frecuencia(segmento_evento))
    caracteristicas.update(caracteristicas_wavelet(segmento_evento))
    caracteristicas.update(caracteristicas_envolvente(segmento_evento))

    # La lista final de características que se quiere mantener
    caracteristicas_top_1_requeridas = [
        'curtosis', 'env_media', 'env_std', 'factor_cresta', 'fft_energia', 
        'profundidad_relativa', 'rms_segmento', 'wavelet_energia_cD1', 
        'wavelet_energia_cD2', 'wavelet_energia_cD3', 'wavelet_energia_cD4', 
        'wavelet_entropia_cD1', 'wavelet_entropia_cD2', 'wavelet_entropia_cD3', 
        'wavelet_entropia_cD4', 'wavelet_std_cD1','wavelet_std_cD2','wavelet_std_cD3','wavelet_std_cD4'
    ]
    
    # Filtrar el diccionario para devolver solo las claves requeridas
    caracteristicas_filtradas = {clave: valor for clave, valor in caracteristicas.items() if clave in caracteristicas_top_1_requeridas}
    
    return caracteristicas_filtradas
# =============================================================================
# FUNCIONES DE PROCESamiento TRIFÁSICO
# =============================================================================

def calcular_componentes_simetricas(va, vb, vc):
    """
    Calcula las componentes de secuencia positiva (V1) y negativa (V2).
    """
    a = np.exp(1j * 2 * np.pi / 3) # Operador de fase 'a'
    
    # Convertir a números complejos para el cálculo
    va_c, vb_c, vc_c = va.astype(complex), vb.astype(complex), vc.astype(complex)
    
    # Fórmulas de Fortescue para V1 y V2
    v1 = (1/3) * (va_c + a * vb_c + a**2 * vc_c)
    v2 = (1/3) * (va_c + a**2 * vb_c + a * vc_c)
    
    return v1, v2

def procesar_una_señal(signal_df):
    """
    Función "wrapper" que procesa una señal trifásica completa:
    1. Segmenta el evento.
    2. Calcula componentes simétricas.
    3. Extrae las características.
    """
    etiqueta = signal_df['etiqueta'].iloc[0]
    va_orig = signal_df['Phase_A'].values
    vb_orig = signal_df['Phase_B'].values
    vc_orig = signal_df['Phase_C'].values
    fs = 7200
    
    # Calcular RMS nominal de los primeros ciclos de la señal sin perturbar
    # (Necesario para la característica 'profundidad_relativa')
    ciclos_nominales = 5
    muestras_ciclo = ad.muestras_por_ciclo(fs) # Asume que esta función está definida
    muestras_nominales = ciclos_nominales * muestras_ciclo
    rms_nominal_val = np.sqrt(np.mean(va_orig[:muestras_nominales]**2))

    # 1. Segmentar el evento para encontrar el inicio y fin
    # Asume que 'encontrar_intervalo_global' está definida previamente
    t_inicio, t_fin = ad.encontrar_intervalo_global(va_orig, vb_orig, vc_orig, fs)
    
    # Si se detecta un evento, se usan los segmentos. Si no, la señal completa.
    if t_inicio is not None and t_fin is not None:
        va_seg, vb_seg, vc_seg = va_orig[t_inicio:t_fin], vb_orig[t_inicio:t_fin], vc_orig[t_inicio:t_fin]
    else:
        va_seg, vb_seg, vc_seg = va_orig, vb_orig, vc_orig

    # 2. Calcular componentes simétricas
    v1, v2 = calcular_componentes_simetricas(va_seg, vb_seg, vc_seg)
    
    # 3. Extraer características usando TUS funciones
    features_v1 = extraer_caracteristicas(np.abs(v1), fs, rms_nominal_val)
    features_v2 = extraer_caracteristicas(np.abs(v2), fs, rms_nominal_val)
    
    # 4. Organizar el resultado final en una fila
    fila_final = {'etiqueta': etiqueta}
    for key, value in features_v1.items():
        fila_final[f'{key}_V1'] = value
    for key, value in features_v2.items():
        fila_final[f'{key}_V2'] = value
        
    # Crear la Serie de Pandas y asignarle un nombre (importante para Dask)
    result_series = pd.Series(fila_final)
    result_series.name = signal_df.name
    
    return result_series

In [8]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import time
import os
import glob


if __name__ == '__main__':
    start_time = time.time()

    input_path = r"C:\Users\Juan_Saa\Documents\Juan_Saa\Proyecto_Grado\Power-Quality\Generacion_Señales\Trifasico\Dataset_Trifasico_Completo_2.parquet"
    output_path = "Dataset_Caracteristicas_Trifasico.parquet"
    temp_dir = "temp_caracteristicas"

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)

    print("Obteniendo la lista de clases del dataset...")
    clases_de_fallas = pd.read_parquet(input_path, columns=['etiqueta'])['etiqueta'].unique()
    print(f"Clases encontradas: {clases_de_fallas}")

    # --- CORRECCIÓN EN EL ORDEN DE LA LISTA BASE ---
    # El orden debe ser idéntico a como se generan en las funciones de extracción.
    # Primero las de tiempo, luego frecuencia, luego wavelet (intercalado), y finalmente envolvente.
    caracteristicas_base = [
        'rms_segmento', 'profundidad_relativa', 'factor_cresta', 'curtosis',
        'fft_energia',
        'wavelet_energia_cD1', 'wavelet_entropia_cD1', 'wavelet_std_cD1',
        'wavelet_energia_cD2', 'wavelet_entropia_cD2', 'wavelet_std_cD2',
        'wavelet_energia_cD3', 'wavelet_entropia_cD3', 'wavelet_std_cD3',
        'wavelet_energia_cD4', 'wavelet_entropia_cD4', 'wavelet_std_cD4',
        'env_media', 'env_std'
    ]
    
    # Esta parte para generar el meta sigue igual, pero ahora usará la lista corregida
    meta = {'etiqueta': 'object'}
    # Primero, añadimos todas las columnas _V1, en orden
    for feature in caracteristicas_base:
        meta[f'{feature}_V1'] = 'float64'
    # Después, añadimos todas las columnas _V2, en orden
    for feature in caracteristicas_base:
        meta[f'{feature}_V2'] = 'float64'
    # --- FIN DE LA CORRECCIÓN ---

    for clase in clases_de_fallas:
        print(f"\n--- Iniciando procesamiento para la clase: {clase} ---")
        ddf_clase = dd.read_parquet(input_path, filters=[('etiqueta', '=', clase)])
        
        with ProgressBar():
            caracteristicas_df = ddf_clase.groupby('signal_id').apply(procesar_una_señal, meta=meta).compute()
        
        temp_file_path = os.path.join(temp_dir, f"caracteristicas_{clase}.parquet")
        caracteristicas_df.to_parquet(temp_file_path, index=True)
        print(f"Características para la clase '{clase}' guardadas en: {temp_file_path}")

    print("\n--- Uniendo todos los archivos de características ---")
    ddf_final = dd.read_parquet(os.path.join(temp_dir, "*.parquet"))
    
    with ProgressBar():
        ddf_final.to_parquet(output_path, write_index=True)

    print(f"\nProcesamiento completado.")
    end_time = time.time()
    print(f"¡Proceso completo en {(end_time - start_time) / 60:.2f} minutos!")
    print(f"Dataset de características final guardado en: {output_path}")

Obteniendo la lista de clases del dataset...
Clases encontradas: ['Normal' 'Sag_A' 'Sag_B' 'Sag_C' 'Sag_D' 'Sag_E' 'Sag_F' 'Sag_G']

--- Iniciando procesamiento para la clase: Normal ---
[################################        ] | 80% Completed | 42.19 ss


ValueError: The columns in the computed data do not match the columns in the provided metadata.
Order of columns does not match.
Actual:   ['etiqueta', 'rms_segmento_V1', 'profundidad_relativa_V1', 'factor_cresta_V1', 'curtosis_V1', 'fft_energia_V1', 'wavelet_energia_cD1_V1', 'wavelet_entropia_cD1_V1', 'wavelet_std_cD1_V1', 'wavelet_energia_cD2_V1', 'wavelet_entropia_cD2_V1', 'wavelet_std_cD2_V1', 'wavelet_energia_cD3_V1', 'wavelet_entropia_cD3_V1', 'wavelet_std_cD3_V1', 'wavelet_energia_cD4_V1', 'wavelet_entropia_cD4_V1', 'wavelet_std_cD4_V1', 'env_media_V1', 'env_std_V1', 'rms_segmento_V2', 'profundidad_relativa_V2', 'factor_cresta_V2', 'curtosis_V2', 'fft_energia_V2', 'wavelet_energia_cD1_V2', 'wavelet_entropia_cD1_V2', 'wavelet_std_cD1_V2', 'wavelet_energia_cD2_V2', 'wavelet_entropia_cD2_V2', 'wavelet_std_cD2_V2', 'wavelet_energia_cD3_V2', 'wavelet_entropia_cD3_V2', 'wavelet_std_cD3_V2', 'wavelet_energia_cD4_V2', 'wavelet_entropia_cD4_V2', 'wavelet_std_cD4_V2', 'env_media_V2', 'env